# GeoAgent — Autonomous Geospatial AI

Natural language → complete geospatial pipeline. No tool names needed.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('.').parent))
import pygeovision as pgv
from pygeovision.agent import (
    GeoAgent, HeuristicPlanner, LLMPlanner,
    Plan, Step, PlanExecutor, GeoAgentMemory, build_tools, TOOL_REGISTRY,
)
print(f"PyGeoVision {pgv.__version__}")
print(f"GeoAgent tools: {list(TOOL_REGISTRY.keys())}")

## Decision hierarchy: how the agent reads the problem

In [ ]:
# The agent makes 3 decisions in order from the problem description:
#   1. SENSOR  — SAR vs optical
#   2. TASK    — flood / damage / oil_spill / land_cover / NDVI / ...
#   3. APPROACH — exact tool sequence + rationale

from pygeovision.agent.planner import HeuristicPlanner
from pygeovision.agent.tools import build_tools
from unittest.mock import MagicMock

tools   = build_tools(MagicMock())
planner = HeuristicPlanner(tools)
ctx     = {"bbox": [-0.3, 5.5, -0.05, 5.7], "date": "2026-06", "output_dir": "/tmp/"}

queries = [
    "Map SAR flood extent — cloud cover 100%, no optical available",
    "Detect building damage after earthquake using Sentinel-1 SAR",
    "Map flood using Sentinel-2 optical imagery — clear sky today",
    "Detect oil spill with Sentinel-1 at night",
    "Monitor land subsidence / ground sinking in Jakarta",
    "Classify land cover using Prithvi foundation model",
    "Compute NDVI vegetation health index",
    "Extract building footprints in Dubai",
    "Map wildfire burn scars after the California fire",
    "Detect ships in the Strait of Hormuz",
]

print(f"{'Query':<58} {'Sensor':>8}  {'Task':>12}  First tool")
print("-" * 105)
for q in queries:
    p = planner.plan(q, context=ctx)
    first_tool = p.steps[0].tool_name if p.steps else "N/A"
    print(f"{q[:57]:<58} {p.sensor:>8}  {p.task:>12}  {first_tool}")

## Running the agent

In [ ]:
# client = pgv.PyGeoVision()
# agent  = GeoAgent(
#     client,
#     output_dir  = "./agent_results/",
#     verbose     = True,    # prints step-by-step progress
# )
# 
# # Set spatial context once
# agent.set_context(
#     bbox=[-0.30, 5.50, -0.05, 5.70],   # Accra, Ghana
#     date="2026-06",
# )
# 
# # Natural language query → complete pipeline
# trace = agent.run("Map current flood inundation in the Odaw River Basin")
# print(trace.final_output)     # → ./agent_results/flood_mask.geojson
# print(trace.summary())        # step-by-step execution log
print("Agent usage:")
print("  agent = GeoAgent(client, output_dir='./results/')")
print("  agent.set_context(bbox=[lon_min, lat_min, lon_max, lat_max], date='2026-06')")
print("  trace = agent.run('natural language query')")
print("  print(trace.final_output)")

## Streaming progress

In [ ]:
# Real-time step events for UIs and logging
# 
# for event in agent.stream("Classify land cover across Ethiopia"):
#     if event["type"] == "plan":
#         print(f"Plan: {event['steps']} steps ({event['planner']})")
#     elif event["type"] == "step_start":
#         print(f"  [{event['step']}] ▶ {event['tool']}")
#     elif event["type"] == "step_done":
#         ok = "✓" if event["success"] else "✗"
#         print(f"  [{event['step']}] {ok} {event['tool']} ({event['duration_s']:.1f}s)")
#     elif event["type"] == "execution_complete":
#         print(f"Done: {event['final_output']}")
print("Streaming: agent.stream('query') yields typed event dicts:")
print("  {'type': 'plan',               'steps': 4, 'planner': 'heuristic'}")
print("  {'type': 'step_start',         'step': 0, 'tool': 'search_satellite_data'}")
print("  {'type': 'step_done',          'step': 0, 'success': True, 'duration_s': 2.1}")
print("  {'type': 'execution_complete', 'success': True, 'final_output': '...'}")

## LLM planning (Anthropic API)

In [ ]:
# With an API key, Claude decomposes the query into an optimal plan
# 
# import os
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."
# 
# # GeoAgent auto-detects the key and uses claude-sonnet-4-6
# agent_llm = GeoAgent(client, verbose=True)
# 
# # Complex multi-step query handled by Claude
# trace = agent_llm.run(
#     "Compare deforestation rates in the Amazon between 2022 and 2024, "
#     "generate a change map with area statistics, and flag any areas "
#     "where loss rate exceeds 5% per year"
# )
print("LLM planning: set ANTHROPIC_API_KEY environment variable")
print("  Agent auto-detects → uses claude-sonnet-4-6 for query decomposition")
print("  Heuristic mode (no key) covers ~80% of common workflows")

## Memory and session persistence

In [ ]:
# Memory: spatial context + turn history + named bindings
# 
# agent.set_context(bbox=[...], date="2026-06")
# agent.bind("flood_mask", "./results/flood_mask.tif")
# 
# # Multi-turn conversation
# agent.run("Map flood extent")
# agent.run("Now show the historical flood frequency for this area")  # uses context
# 
# # Save/restore session
# agent.save_session("./sessions/accra_session.json")
# agent.load_session("./sessions/accra_session.json")
# 
# # History
# for turn in agent.history():
#     print(f"Turn {turn['turn']}: {turn['query']}")

from pygeovision.agent.memory import GeoAgentMemory
mem = GeoAgentMemory()
mem.set_context(bbox=[-0.3, 5.5, -0.05, 5.7], date="2026-06")
mem.bind("flood_mask", "/tmp/flood.tif")
print(mem)
print("Context:", mem.context_for_planner())

## Summary

### GeoAgent decision matrix

| Sensor | Task | Tool sequence |
|---|---|---|
| SAR | flood | `sar_preprocess → sar_flood_detection → postprocess` |
| SAR | damage | `sar_preprocess → change_detection(4-class) → postprocess` |
| SAR | oil_spill | `sar_preprocess → sar_flood_detection(dark-pixel) → postprocess` |
| SAR | subsidence | `sar_preprocess → change_detection(amplitude) → postprocess` |
| optical | flood | `search → download → prepare → prithvi(flood) → postprocess` |
| optical | land_cover | `search → download → prepare → prithvi(land_cover) → postprocess` |
| optical | NDVI | `search → download → prepare → compute_spectral_index → postprocess` |
| optical | building | `run_pipeline(building_footprints)` |

**Two planning modes:**
- **Heuristic** (no API key): keyword scoring, zero external calls
- **LLM** (ANTHROPIC_API_KEY): Claude reads full problem description + tool schemas